# Imports

In [17]:
import os
import sys
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import PIL

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision import models
import cv2
from torch.utils.data import WeightedRandomSampler

from PIL import Image, ImageOps

print(f"PyTorch version: {torch.__version__}")
print(f"OpenCV version: {cv2.__version__}")
print(f"Numpy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Pillow (PIL) version: {PIL.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRAIN_DIR = '/kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge/train'
TEST_DIR = '/kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge/test'
train_df = pd.read_csv('/kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge/train_metadata.csv')
test_df = pd.read_csv('/kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge/test_metadata.csv')

PyTorch version: 2.10.0+cu128
OpenCV version: 4.13.0
Numpy version: 2.0.2
Pandas version: 2.3.3
Pillow (PIL) version: 11.3.0
Scikit-learn version: 1.6.1


# Encodage

In [2]:
# Encodage avec LabelEncoder
encoder = LabelEncoder()
train_df['label_encoded'] = encoder.fit_transform(train_df['label'])
nb_cl = len(encoder.classes_)
print(nb_cl)

# Sauvegarde par sécurité
np.save('classes.npy', encoder.classes_)
print("Encodeur sauvegardé")


13
Encodeur sauvegardé


# Pré-traitements et création des Datasets

In [3]:
def apply_watershed_mask(image_pil):
    # watershed OpenCV donc on convertit
    img = np.array(image_pil.convert('RGB'))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    # Otsu: deux classes qui minimisent la variance intra-classes
    ret, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    kernel = np.ones((3,3), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)

    # masque dilaté pour ne pas couper d'éléments en trop
    cell_mask = cv2.dilate(opening, kernel, iterations=2)
    background_mask = cv2.bitwise_not(cell_mask) # donc tout sauf le masque
    img_final = img.astype(np.float32)
    alpha = 0.5
    img_final[background_mask == 255] *= alpha #application du masque

    return Image.fromarray(img_final.astype(np.uint8)).convert("RGB")

In [4]:
class TrainDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['ID']
        img_path = os.path.join(self.img_dir, img_id)

        # Envoi de l'image
        image = Image.open(img_path).convert("RGB")
        # pré-traitements (fonction plus haut)
        image = apply_watershed_mask(image)
        img_np = np.array(image)
        lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)) # CLAHE seulement sur la luminosité (lab)
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        img_np = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        image = Image.fromarray(img_np)
        image = ImageOps.autocontrast(image) # autocontrast

        # Envoi du label
        label = int(self.df.iloc[idx]['label_encoded'])

        # Data augmentation + autres transforms (ci-dessous)
        if self.transform:
            image = self.transform(image)

        return image, label

In [5]:
train_tf= transforms.Compose([
    transforms.Resize((288, 288)), # plus de détails pour convxnet
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

#idem sauf data augmentation
test_tf = transforms.Compose([
    transforms.Resize((288, 288)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


train_nb = int(0.8 * len(train_df)) #split pour éviter l'overfitting et voir dans les logs l'évolutioon
indices = np.arange(len(train_df))
np.random.seed(42) # Pour que le split soit reproductible
np.random.shuffle(indices)
train_idx, val_idx = indices[:train_nb], indices[train_nb:]

train_dataset = TrainDataset(train_df.iloc[train_idx], TRAIN_DIR, transform=train_tf)

val_dataset = TrainDataset(train_df.iloc[val_idx], TRAIN_DIR, transform=test_tf)


train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [6]:
class TestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['ID']
        img_path = os.path.join(self.img_dir, f"{img_id}")

        # Pareil que Train juste on a pas les labels
        image = Image.open(img_path).convert("RGB")
        image = apply_watershed_mask(image)
        img_np = np.array(image)
        lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        img_np = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        image = Image.fromarray(img_np)
        image = ImageOps.autocontrast(image)

        if self.transform:
            image = self.transform(image)

        return image

# Modèle

In [7]:
def get_convnext_model(num_classes, freeze_backbone=True):
    model = models.convnext_tiny(weights='IMAGENET1K_V1') #importation

    if freeze_backbone: #pour train la tête mieux au début
        for param in model.features.parameters():
            param.requires_grad = False

    in_features = model.classifier[2].in_features

    #changements
    model.classifier[2] = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(512, nb_cl)     # nb: output de probas
    )
    return model


In [8]:
model = get_convnext_model(num_classes=13).to(device)

train_labels = train_dataset.df['label_encoded'].values

class_sample_count = np.array([len(np.where(train_labels == t)[0]) for t in np.unique(train_labels)])
weight = 1. / class_sample_count
samples_weight = torch.from_numpy(np.array([weight[t] for t in train_labels])).float()

sampler = WeightedRandomSampler(weights=samples_weight, num_samples=len(samples_weight), replacement=True)

# mis à jour (en soit val ne change pas)
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 200MB/s]  


# Entrainement

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
model = get_convnext_model(num_classes=13, freeze_backbone=True).to(device)
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=0.01) # nb: optimizer pr la tête
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.2, patience=3)

patience = 10
best_val_f1 = 0.0
counter = 0
epochs = 40
unfrozen = False

for epoch in range(epochs):
    if epoch >= 3 and not unfrozen: # tête
        for param in model.features.parameters():
            param.requires_grad = True
        optimizer = torch.optim.AdamW([
            {'params': model.features.parameters(), 'lr': 1e-5},
            {'params': model.classifier.parameters(), 'lr': 1e-4}
        ], weight_decay=0.05)

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.2, patience=2)
        unfrozen = True
    model.train()
    train_loss = 0.0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # validation
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_train_loss = train_loss/ len(train_loader)
    avg_val_loss = val_loss/ len(val_loader)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    accuracy = 100 * (np.array(all_preds) == np.array(all_labels)).mean()
    #affichage
    print(f"\nEpoch [{epoch+1}/{epochs}]")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"Val Acc: {accuracy:.2f}% | Val F1-Score: {epoch_f1:.4f}")

    scheduler.step(epoch_f1) # pas déterminé par le score

    if epoch_f1 > best_val_f1:
        best_val_f1 = epoch_f1
        torch.save(model.state_dict(), 'best_model.pth')
        counter = 0
        print("Modèle sauvegardé")
    else:
        counter += 1
        if counter >= patience:
            print(f"\nEarly stopping déclenché à l'étape {epoch+1}")
            break

# Prédictions

In [9]:
def get_predictions(test_loader, model, device):
    model.eval()
    all_preds = []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Inférence TTA"):
            imgs = batch.to(device)

            out = F.softmax(model(imgs), dim=1)
            out += F.softmax(model(torch.flip(imgs, dims=[3])), dim=1) #TTA: prédiction sur la cellule un peu déplacée
            out += F.softmax(model(torch.flip(imgs, dims=[2])), dim=1)
            imgs_rot = torch.rot90(imgs, k=1, dims=[2, 3])
            out += F.softmax(model(imgs_rot), dim=1)

            avg = out/4
            pred = avg.argmax(dim=1)
            all_preds.extend(pred.cpu().numpy())

    return np.array(all_preds)

In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
model.to(device)
model.eval()

test_dataset = TestDataset(test_df, TEST_DIR, transform=test_tf)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

preds = get_predictions(test_loader, model, device)
final_labels = encoder.inverse_transform(preds)

submission = pd.DataFrame({
    'ID': test_df['ID'],
    'label': final_labels
})
submission.to_csv('submission.csv', index=False)
print("Fini!")